# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_02 — Event-Driven Backtest Engine

### Research question

How do we build an event-driven backtest engine that models the sequence of market data, signals, orders, fills, latency, partial execution, position accounting, cash, costs, and audit trails?

This notebook follows:

```text
05_01_vectorized_backtest_engine
```

The previous notebook built a vectorized backtest engine. That is fast and ideal for daily strategies where fills can be approximated by lagged weights.

This notebook builds an **event-driven engine**, which is closer to production trading simulation.

It covers:

1. event-driven architecture;
2. market data events;
3. signal events;
4. order events;
5. fill events;
6. event queue;
7. latency;
8. market orders;
9. limit orders;
10. partial fills;
11. L1 bid/ask execution;
12. slippage and commission;
13. cash and position ledger;
14. realised and unrealised PnL;
15. portfolio valuation;
16. risk checks;
17. order rejection rules;
18. benchmark comparison;
19. fill-quality analytics;
20. reconciliation against a vectorized approximation;
21. event log and audit trail;
22. governance flags;
23. saved outputs and manifest.

The key idea:

> A vectorized backtest asks “what return did the portfolio earn?” An event-driven backtest asks “what happened first, what did we know, what did we send, what got filled, and what did it cost?”

## 1. Why event-driven backtesting?

Vectorized backtests are excellent for research speed.

But they usually simplify:

- execution timing;
- intraday bid/ask;
- order latency;
- partial fills;
- cancelled orders;
- cash constraints;
- fill-level commission;
- market impact;
- order book state;
- position updates.

An event-driven backtest models the trading process as a sequence of events:

```text
MarketDataEvent -> SignalEvent -> OrderEvent -> FillEvent -> PortfolioUpdate
```

This structure is slower but more faithful to trading reality.

## 2. Event types

We use four main event types:

### MarketDataEvent

New price, bid/ask, volume, and timestamp information.

### SignalEvent

Strategy decides what it wants to trade.

### OrderEvent

Portfolio/risk engine converts desired trade into an order.

### FillEvent

Execution simulator fills all or part of the order.

Each event is timestamped and logged.

This makes the backtest auditable.

## 3. Execution timing

A typical event-driven loop:

1. read latest market data;
2. update portfolio mark-to-market;
3. strategy receives market data and emits signals;
4. order manager turns signals into orders;
5. execution simulator applies latency and liquidity constraints;
6. fills update positions and cash;
7. risk and performance are recorded.

This avoids the common research mistake of using prices that were not yet available.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from collections import deque
import json
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class EventBacktestConfig:
    n_days: int = 45
    minutes_per_day: int = 78
    seed: int = 42
    initial_cash: float = 1_000_000.0
    annualisation: int = 252
    rebalance_minutes: int = 13
    signal_lookback: int = 30
    max_gross_exposure: float = 1.25
    max_single_name_notional: float = 250_000.0
    target_gross_exposure: float = 0.90
    order_latency_minutes: int = 1
    participation_rate: float = 0.08
    market_order_spread_fraction: float = 0.55
    limit_order_spread_fraction: float = 0.15
    commission_bps: float = 0.35
    min_commission: float = 0.50
    borrow_rate_ann: float = 0.025
    cash_rate_ann: float = 0.020
    leverage_financing_rate_ann: float = 0.045
    use_limit_orders: bool = True
    limit_order_ttl_minutes: int = 3

config = EventBacktestConfig()
config

## 5. Synthetic intraday L1 data

We simulate intraday top-of-book data for four liquid instruments.

For each asset and timestamp:

- mid price;
- bid;
- ask;
- spread;
- available bid/ask size;
- traded volume;
- regime state.

This is still simplified, but it is much closer to execution than daily close-to-close returns.

In [ ]:
def simulate_intraday_l1_data(config: EventBacktestConfig):
    rng = np.random.default_rng(config.seed)

    assets = ["US_EQ", "US_BOND", "GOLD", "OIL"]
    asset_class = {
        "US_EQ": "equity",
        "US_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
    }

    start_prices = pd.Series({
        "US_EQ": 100.0,
        "US_BOND": 100.0,
        "GOLD": 100.0,
        "OIL": 100.0,
    })

    base_spread_bps = pd.Series({
        "US_EQ": 1.2,
        "US_BOND": 0.8,
        "GOLD": 2.0,
        "OIL": 3.0,
    })

    base_minute_vol = pd.Series({
        "US_EQ": 0.0012,
        "US_BOND": 0.00045,
        "GOLD": 0.0010,
        "OIL": 0.0018,
    })

    base_volume = pd.Series({
        "US_EQ": 20_000,
        "US_BOND": 35_000,
        "GOLD": 12_000,
        "OIL": 10_000,
    })

    total_minutes = config.n_days * config.minutes_per_day
    dates = pd.bdate_range("2024-01-02", periods=config.n_days)

    timestamps = []
    for day in dates:
        session_start = pd.Timestamp(day.date()) + pd.Timedelta(hours=9, minutes=30)
        for m in range(config.minutes_per_day):
            timestamps.append(session_start + pd.Timedelta(minutes=5 * m))

    index = pd.DatetimeIndex(timestamps, name="timestamp")

    factor_names = ["market", "rates", "commodity"]
    factor_corr = np.array([
        [1.00, -0.25, 0.25],
        [-0.25, 1.00, 0.05],
        [0.25, 0.05, 1.00],
    ])

    factor_vol = np.array([0.0010, 0.00045, 0.0012])
    factor_cov = np.outer(factor_vol, factor_vol) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc["US_EQ"] = [1.0, -0.15, 0.10]
    loadings.loc["US_BOND"] = [-0.20, 1.0, 0.05]
    loadings.loc["GOLD"] = [0.05, 0.25, 0.65]
    loadings.loc["OIL"] = [0.35, 0.00, 1.15]

    regime = np.zeros(total_minutes, dtype=int)
    stress_type = np.array(["normal"] * total_minutes, dtype=object)

    prices = pd.DataFrame(index=index, columns=assets, dtype=float)
    returns = pd.DataFrame(index=index, columns=assets, dtype=float)
    spreads_bps = pd.DataFrame(index=index, columns=assets, dtype=float)
    bid = pd.DataFrame(index=index, columns=assets, dtype=float)
    ask = pd.DataFrame(index=index, columns=assets, dtype=float)
    bid_size = pd.DataFrame(index=index, columns=assets, dtype=float)
    ask_size = pd.DataFrame(index=index, columns=assets, dtype=float)
    volume = pd.DataFrame(index=index, columns=assets, dtype=float)

    current_price = start_prices.copy()

    for t, ts in enumerate(index):
        if t > 0:
            if regime[t - 1] == 0:
                regime[t] = rng.choice([0, 1], p=[0.992, 0.008])
            else:
                regime[t] = rng.choice([0, 1], p=[0.08, 0.92])

        vol_mult = 1.0 if regime[t] == 0 else 2.8
        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov * vol_mult**2)

        u = rng.uniform()
        if u < 0.0025:
            stress_type[t] = "risk_off"
            f[0] -= rng.uniform(0.008, 0.025)
            f[1] += rng.uniform(0.001, 0.006)
            f[2] -= rng.uniform(0.002, 0.010)
        elif u < 0.0045:
            stress_type[t] = "commodity_spike"
            f[2] += rng.uniform(0.006, 0.020)
            f[0] -= rng.uniform(0.001, 0.008)

        eps = rng.standard_normal(len(assets)) * base_minute_vol.values * vol_mult * 0.35
        ret = loadings.to_numpy() @ f + eps
        current_price = current_price * np.exp(ret)

        returns.iloc[t] = ret
        prices.iloc[t] = current_price.values

        spread_noise = rng.lognormal(mean=0.0, sigma=0.25, size=len(assets))
        spread = base_spread_bps.values * spread_noise * (1.0 + 0.75 * regime[t])
        spreads_bps.iloc[t] = spread

        half_spread = current_price.values * spread / 10000.0 / 2.0
        bid.iloc[t] = current_price.values - half_spread
        ask.iloc[t] = current_price.values + half_spread

        vol_noise = rng.lognormal(mean=0.0, sigma=0.60, size=len(assets))
        minute_volume = base_volume.values * vol_noise * (1.0 + 0.80 * regime[t])
        volume.iloc[t] = minute_volume

        bid_size.iloc[t] = minute_volume * rng.uniform(0.08, 0.25, size=len(assets))
        ask_size.iloc[t] = minute_volume * rng.uniform(0.08, 0.25, size=len(assets))

    l1 = {}
    for asset in assets:
        l1[asset] = pd.DataFrame({
            "mid": prices[asset],
            "bid": bid[asset],
            "ask": ask[asset],
            "spread_bps": spreads_bps[asset],
            "bid_size": bid_size[asset],
            "ask_size": ask_size[asset],
            "volume": volume[asset],
            "return": returns[asset],
        }, index=index)

    regime_info = pd.DataFrame({
        "regime": regime,
        "stress_type": stress_type,
    }, index=index)

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "base_spread_bps": [base_spread_bps[a] for a in assets],
        "base_minute_vol": [base_minute_vol[a] for a in assets],
        "base_volume": [base_volume[a] for a in assets],
    })

    return l1, prices, returns, regime_info, metadata

l1, prices, returns, regime_info, metadata = simulate_intraday_l1_data(config)
assets = metadata["asset"].tolist()

prices.head(), metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in assets:
    plt.plot(prices.index, prices[asset], label=asset)
plt.title("Synthetic Intraday Mid Prices")
plt.xlabel("Timestamp")
plt.ylabel("Mid price")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(regime_info.index, regime_info["regime"], drawstyle="steps-post")
plt.title("Intraday Regime Indicator")
plt.xlabel("Timestamp")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Event and accounting data structures

The engine uses lightweight dictionaries for events to keep the logic transparent.

Each event has:

- event type;
- timestamp;
- asset if relevant;
- payload fields;
- unique IDs for orders and fills.

In [ ]:
def make_event(event_type, timestamp, **kwargs):
    event = {
        "event_type": event_type,
        "timestamp": timestamp,
    }
    event.update(kwargs)
    return event

order_id_counter = itertools.count(1)
fill_id_counter = itertools.count(1)

## 7. Strategy: intraday momentum with defensive regime tilt

The strategy observes recent mid-price returns.

Every `rebalance_minutes`, it computes a cross-sectional momentum score:

$$
score_i = z\Big(\sum_{j=1}^{L} r_{i,t-j}\Big)
$$

It then targets:

- long positive momentum assets;
- short negative momentum assets;
- smaller gross exposure in stress regimes.

This is not intended as a production alpha. It is a test strategy for the engine.

In [ ]:
def cross_sectional_zscore_vector(x):
    x = pd.Series(x)
    std = x.std()
    if std == 0 or not np.isfinite(std):
        return x * 0.0
    return (x - x.mean()) / std

def generate_signal_weights(current_time, history_prices, config, current_regime):
    if len(history_prices) < config.signal_lookback + 1:
        return pd.Series(0.0, index=history_prices.columns)

    lookback_returns = history_prices.pct_change(config.signal_lookback).iloc[-1]
    score = cross_sectional_zscore_vector(lookback_returns).reindex(history_prices.columns).fillna(0.0)

    raw = score - score.mean()
    if raw.abs().sum() == 0:
        return pd.Series(0.0, index=history_prices.columns)

    gross = config.target_gross_exposure
    if current_regime == 1:
        gross *= 0.55

    weights = raw / raw.abs().sum() * gross
    weights = weights.clip(lower=-config.max_single_name_notional / config.initial_cash,
                           upper=config.max_single_name_notional / config.initial_cash)

    if weights.abs().sum() > 0:
        weights = weights / weights.abs().sum() * min(gross, weights.abs().sum())

    return weights.fillna(0.0)

sample_signal = generate_signal_weights(prices.index[100], prices.iloc[:101], config, current_regime=0)
sample_signal

## 8. Portfolio state

The portfolio state tracks:

- cash;
- positions in shares;
- last prices;
- realised cashflows;
- orders;
- fills;
- NAV history.

The state is updated only through fills and mark-to-market updates.

In [ ]:
class PortfolioState:
    def __init__(self, initial_cash, assets):
        self.initial_cash = float(initial_cash)
        self.cash = float(initial_cash)
        self.positions = pd.Series(0.0, index=assets)
        self.last_mid = pd.Series(np.nan, index=assets)
        self.event_log = []
        self.order_log = []
        self.fill_log = []
        self.nav_log = []

    def update_market(self, timestamp, mid_prices):
        self.last_mid.loc[mid_prices.index] = mid_prices

    def market_value(self):
        return float((self.positions * self.last_mid.fillna(0.0)).sum())

    def gross_exposure(self):
        return float((self.positions * self.last_mid.fillna(0.0)).abs().sum())

    def net_liquidation_value(self):
        return float(self.cash + self.market_value())

    def weights(self):
        nav = self.net_liquidation_value()
        if nav <= 0:
            return pd.Series(0.0, index=self.positions.index)
        return self.positions * self.last_mid.fillna(0.0) / nav

    def record_nav(self, timestamp, regime=None, stress_type=None):
        nav = self.net_liquidation_value()
        mv = self.market_value()
        gross = self.gross_exposure()
        w = self.weights()
        self.nav_log.append({
            "timestamp": timestamp,
            "cash": self.cash,
            "market_value": mv,
            "nav": nav,
            "gross_exposure": gross,
            "net_exposure": float(w.sum()),
            "long_exposure": float(w.clip(lower=0.0).sum()),
            "short_exposure": float(w.clip(upper=0.0).sum()),
            "regime": regime,
            "stress_type": stress_type,
        })

    def apply_fill(self, fill):
        asset = fill["asset"]
        quantity = fill["fill_quantity"]
        price = fill["fill_price"]
        commission = fill["commission"]

        cash_delta = -quantity * price - commission
        self.cash += cash_delta
        self.positions.loc[asset] += quantity
        self.fill_log.append(fill)

    def log_order(self, order):
        self.order_log.append(order)

    def log_event(self, event):
        self.event_log.append(event)

## 9. Order generation and risk checks

The order manager converts target weights into share orders.

It applies basic risk constraints:

- max gross exposure;
- max single-name notional;
- cash sanity checks;
- no trading if NAV is invalid.

In [ ]:
def create_orders_from_target_weights(timestamp, portfolio, target_weights, market_snapshot, config):
    nav = portfolio.net_liquidation_value()
    if nav <= 0:
        return []

    current_weights = portfolio.weights().reindex(target_weights.index).fillna(0.0)
    desired_notional = target_weights * nav
    current_notional = current_weights * nav
    trade_notional = desired_notional - current_notional

    orders = []

    projected_gross = target_weights.abs().sum()
    if projected_gross > config.max_gross_exposure + 1e-12:
        scale = config.max_gross_exposure / projected_gross
        trade_notional *= scale

    for asset, notional in trade_notional.items():
        mid = market_snapshot.loc[asset, "mid"]
        if not np.isfinite(mid) or mid <= 0:
            continue

        if abs(notional) < 1_000:
            continue

        quantity = np.round(notional / mid)

        if quantity == 0:
            continue

        order_type = "LIMIT" if config.use_limit_orders else "MARKET"

        if quantity > 0:
            side = "BUY"
            limit_price = market_snapshot.loc[asset, "bid"] + config.limit_order_spread_fraction * (market_snapshot.loc[asset, "ask"] - market_snapshot.loc[asset, "bid"])
        else:
            side = "SELL"
            limit_price = market_snapshot.loc[asset, "ask"] - config.limit_order_spread_fraction * (market_snapshot.loc[asset, "ask"] - market_snapshot.loc[asset, "bid"])

        order = make_event(
            "ORDER",
            timestamp,
            order_id=next(order_id_counter),
            asset=asset,
            side=side,
            quantity=float(quantity),
            remaining_quantity=float(quantity),
            order_type=order_type,
            limit_price=float(limit_price),
            creation_time=timestamp,
            eligible_time=timestamp + pd.Timedelta(minutes=5 * config.order_latency_minutes),
            expiry_time=timestamp + pd.Timedelta(minutes=5 * (config.order_latency_minutes + config.limit_order_ttl_minutes)),
            status="NEW",
            reason="target_weight_rebalance",
        )
        orders.append(order)

    return orders

## 10. Execution simulator

The execution simulator fills orders using current L1 bid/ask data.

### Market order

- buys near ask;
- sells near bid;
- fills up to participation rate times displayed/available volume.

### Limit order

- buy fills if limit price crosses ask;
- sell fills if limit price crosses bid;
- otherwise remains open until expiry.

This simplified model still captures latency, order type, partial fill, and liquidity constraints.

In [ ]:
def calculate_commission(notional, config):
    return max(config.min_commission, abs(notional) * config.commission_bps / 10000.0)

def try_execute_order(order, timestamp, market_snapshot, config):
    asset = order["asset"]

    if timestamp < order["eligible_time"]:
        return None, order

    if timestamp > order["expiry_time"]:
        order = order.copy()
        order["status"] = "EXPIRED"
        return None, order

    row = market_snapshot.loc[asset]
    bid = row["bid"]
    ask = row["ask"]
    volume = row["volume"]
    bid_size = row["bid_size"]
    ask_size = row["ask_size"]

    remaining = order["remaining_quantity"]
    side = order["side"]

    if remaining == 0:
        order = order.copy()
        order["status"] = "FILLED"
        return None, order

    if order["order_type"] == "LIMIT":
        if side == "BUY" and order["limit_price"] < ask:
            return None, order
        if side == "SELL" and order["limit_price"] > bid:
            return None, order

    max_fill_by_volume = max(1.0, config.participation_rate * volume)

    if side == "BUY":
        available = max(1.0, ask_size)
        fill_qty_abs = min(abs(remaining), max_fill_by_volume, available)
        fill_qty = fill_qty_abs
        if order["order_type"] == "MARKET":
            fill_price = ask + config.market_order_spread_fraction * (ask - bid)
        else:
            fill_price = min(order["limit_price"], ask)
    else:
        available = max(1.0, bid_size)
        fill_qty_abs = min(abs(remaining), max_fill_by_volume, available)
        fill_qty = -fill_qty_abs
        if order["order_type"] == "MARKET":
            fill_price = bid - config.market_order_spread_fraction * (ask - bid)
        else:
            fill_price = max(order["limit_price"], bid)

    notional = fill_qty * fill_price
    commission = calculate_commission(notional, config)

    fill = make_event(
        "FILL",
        timestamp,
        fill_id=next(fill_id_counter),
        order_id=order["order_id"],
        asset=asset,
        side=side,
        fill_quantity=float(fill_qty),
        fill_price=float(fill_price),
        fill_notional=float(notional),
        commission=float(commission),
        order_type=order["order_type"],
        bid=float(bid),
        ask=float(ask),
        mid=float(row["mid"]),
        spread_bps=float(row["spread_bps"]),
    )

    order = order.copy()
    order["remaining_quantity"] = float(order["remaining_quantity"] - fill_qty)
    if abs(order["remaining_quantity"]) < 1e-9:
        order["remaining_quantity"] = 0.0
        order["status"] = "FILLED"
    else:
        order["status"] = "PARTIALLY_FILLED"

    return fill, order

## 11. Event-driven backtest loop

This is the main loop.

At each timestamp:

1. create market data snapshot;
2. update marks;
3. execute eligible orders;
4. generate signals on rebalance times;
5. create new orders;
6. update cash for borrow/financing;
7. record NAV.

In [ ]:
def market_snapshot_at(timestamp, l1, assets):
    rows = []
    for asset in assets:
        row = l1[asset].loc[timestamp]
        rows.append({
            "asset": asset,
            "mid": row["mid"],
            "bid": row["bid"],
            "ask": row["ask"],
            "spread_bps": row["spread_bps"],
            "bid_size": row["bid_size"],
            "ask_size": row["ask_size"],
            "volume": row["volume"],
            "return": row["return"],
        })
    return pd.DataFrame(rows).set_index("asset")

def accrue_financing_and_borrow(portfolio, timestamp, config):
    nav = portfolio.net_liquidation_value()
    if nav <= 0:
        return {"cash_interest": 0.0, "borrow_cost": 0.0, "financing_cost": 0.0}

    weights = portfolio.weights()
    gross = weights.abs().sum()

    cash_interest = max(portfolio.cash, 0.0) * config.cash_rate_ann / config.annualisation / config.minutes_per_day
    borrow_cost = (weights.clip(upper=0.0).abs().sum() * nav) * config.borrow_rate_ann / config.annualisation / config.minutes_per_day
    financing_cost = max(gross - 1.0, 0.0) * nav * config.leverage_financing_rate_ann / config.annualisation / config.minutes_per_day

    portfolio.cash += cash_interest - borrow_cost - financing_cost

    return {
        "cash_interest": cash_interest,
        "borrow_cost": borrow_cost,
        "financing_cost": financing_cost,
    }

def run_event_driven_backtest(l1, prices, returns, regime_info, metadata, config):
    assets = list(prices.columns)
    portfolio = PortfolioState(config.initial_cash, assets)
    open_orders = []

    financing_log = []

    for t, timestamp in enumerate(prices.index):
        snapshot = market_snapshot_at(timestamp, l1, assets)
        portfolio.update_market(timestamp, snapshot["mid"])

        market_event = make_event("MARKET", timestamp, n_assets=len(assets))
        portfolio.log_event(market_event)

        updated_open_orders = []
        for order in open_orders:
            fill, updated_order = try_execute_order(order, timestamp, snapshot, config)
            if fill is not None:
                portfolio.apply_fill(fill)
            if updated_order["status"] not in ["FILLED", "EXPIRED", "REJECTED"]:
                updated_open_orders.append(updated_order)
            else:
                portfolio.log_order(updated_order)
        open_orders = updated_open_orders

        should_rebalance = (t % config.rebalance_minutes == 0)
        if should_rebalance:
            history = prices.iloc[:t+1]
            current_regime = int(regime_info.loc[timestamp, "regime"])
            target_weights = generate_signal_weights(timestamp, history, config, current_regime)

            signal_event = make_event(
                "SIGNAL",
                timestamp,
                target_weights=target_weights.to_dict(),
                current_regime=current_regime
            )
            portfolio.log_event(signal_event)

            orders = create_orders_from_target_weights(timestamp, portfolio, target_weights, snapshot, config)
            for order in orders:
                portfolio.log_order(order)
                open_orders.append(order)

        financing = accrue_financing_and_borrow(portfolio, timestamp, config)
        financing["timestamp"] = timestamp
        financing_log.append(financing)

        portfolio.record_nav(
            timestamp,
            regime=int(regime_info.loc[timestamp, "regime"]),
            stress_type=regime_info.loc[timestamp, "stress_type"]
        )

    nav = pd.DataFrame(portfolio.nav_log).set_index("timestamp")
    orders = pd.DataFrame(portfolio.order_log)
    fills = pd.DataFrame(portfolio.fill_log)
    events = pd.DataFrame(portfolio.event_log)
    financing = pd.DataFrame(financing_log).set_index("timestamp")

    if not fills.empty:
        fills["signed_notional"] = fills["fill_quantity"] * fills["fill_price"]
        fills["abs_notional"] = fills["signed_notional"].abs()
        fills["effective_spread_cost"] = np.where(
            fills["fill_quantity"] > 0,
            fills["fill_price"] - fills["mid"],
            fills["mid"] - fills["fill_price"]
        )
        fills["effective_spread_bps"] = fills["effective_spread_cost"] / fills["mid"] * 10000.0

    return {
        "portfolio": portfolio,
        "nav": nav,
        "orders": orders,
        "fills": fills,
        "events": events,
        "financing": financing,
    }

event_results = run_event_driven_backtest(l1, prices, returns, regime_info, metadata, config)

event_results["nav"].head(), event_results["fills"].head()

## 12. Event-driven performance analytics

In [ ]:
event_nav = event_results["nav"].copy()
event_nav["return"] = event_nav["nav"].pct_change().fillna(0.0)
event_nav["benchmark_return"] = (0.60 * returns["US_EQ"] + 0.25 * returns["US_BOND"] + 0.15 * returns["GOLD"]).reindex(event_nav.index).fillna(0.0)
event_nav["benchmark_nav"] = config.initial_cash * (1 + event_nav["benchmark_return"]).cumprod()

def intraday_performance_summary(nav_df, name, config):
    r = nav_df["return"].dropna()
    periods_per_year = config.annualisation * config.minutes_per_day

    ann_return = (1 + r).prod() ** (periods_per_year / len(r)) - 1 if len(r) else np.nan
    ann_vol = r.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    nav_scaled = nav_df["nav"] / nav_df["nav"].iloc[0]
    running_max = nav_scaled.cummax()
    drawdown = nav_scaled / running_max - 1
    max_dd = drawdown.min()

    losses = -r
    var95 = np.quantile(losses, 0.95)
    cvar95 = losses[losses >= var95].mean()

    alpha_ann, beta, corr = alpha_beta(r, nav_df["benchmark_return"], periods_per_year)

    return {
        "strategy": name,
        "annualised_return": float(ann_return),
        "annualised_vol": float(ann_vol),
        "sharpe_proxy": float(sharpe),
        "max_drawdown": float(max_dd),
        "hit_rate": float((r > 0).mean()),
        "skew": float(r.skew()),
        "excess_kurtosis": float(r.kurtosis()),
        "VaR95": float(var95),
        "CVaR95": float(cvar95),
        "total_return": float(nav_scaled.iloc[-1] - 1.0),
        "alpha_ann_vs_benchmark": float(alpha_ann),
        "beta_vs_benchmark": float(beta),
        "corr_vs_benchmark": float(corr),
        "mean_gross_exposure": float(nav_df["gross_exposure"].mean() / nav_df["nav"].mean()),
    }

event_performance = pd.DataFrame([intraday_performance_summary(event_nav, "event_driven_strategy", config)])
event_performance

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(event_nav.index, event_nav["nav"], label="event-driven strategy")
plt.plot(event_nav.index, event_nav["benchmark_nav"], label="benchmark", linestyle="--")
plt.title("Event-Driven Strategy NAV")
plt.xlabel("Timestamp")
plt.ylabel("NAV")
plt.legend()
plt.show()

nav_scaled = event_nav["nav"] / event_nav["nav"].iloc[0]
drawdown = nav_scaled / nav_scaled.cummax() - 1

plt.figure(figsize=(12, 5))
plt.plot(drawdown.index, drawdown)
plt.title("Event-Driven Strategy Drawdown")
plt.xlabel("Timestamp")
plt.ylabel("Drawdown")
plt.show()

## 13. Fill analytics

A fill-level report answers:

- how many orders filled?
- how many were partial?
- what was the effective spread paid?
- how much commission was paid?
- which assets were most expensive to trade?

In [ ]:
fills = event_results["fills"].copy()
orders = event_results["orders"].copy()

if fills.empty:
    fill_summary = pd.DataFrame()
    fill_asset_summary = pd.DataFrame()
else:
    fill_summary = pd.DataFrame([{
        "n_fills": len(fills),
        "n_unique_orders_filled": fills["order_id"].nunique(),
        "total_abs_notional": fills["abs_notional"].sum(),
        "total_commission": fills["commission"].sum(),
        "mean_effective_spread_bps": fills["effective_spread_bps"].mean(),
        "p95_effective_spread_bps": fills["effective_spread_bps"].quantile(0.95),
        "max_effective_spread_bps": fills["effective_spread_bps"].max(),
    }])

    fill_asset_summary = (
        fills
        .groupby("asset")
        .agg(
            n_fills=("fill_id", "count"),
            abs_notional=("abs_notional", "sum"),
            total_commission=("commission", "sum"),
            mean_effective_spread_bps=("effective_spread_bps", "mean"),
            p95_effective_spread_bps=("effective_spread_bps", lambda x: x.quantile(0.95)),
        )
        .reset_index()
        .merge(metadata[["asset", "asset_class"]], on="asset", how="left")
    )

fill_summary, fill_asset_summary

In [ ]:
if not fill_asset_summary.empty:
    plot_df = fill_asset_summary.sort_values("mean_effective_spread_bps")
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["asset"], plot_df["mean_effective_spread_bps"])
    plt.title("Mean Effective Spread Paid by Asset")
    plt.xlabel("Effective spread bps")
    plt.ylabel("Asset")
    plt.show()

## 14. Order lifecycle analytics

The order log shows how many orders were:

- new;
- partially filled;
- filled;
- expired;
- rejected.

Because the current order log stores both submissions and terminal states, we summarise terminal status by latest order record.

In [ ]:
def latest_order_status(orders):
    if orders.empty:
        return pd.DataFrame()
    ordered = orders.sort_values(["order_id", "timestamp"])
    latest = ordered.groupby("order_id").tail(1)
    return latest

latest_orders = latest_order_status(orders)

if latest_orders.empty:
    order_status_summary = pd.DataFrame()
else:
    order_status_summary = (
        latest_orders
        .groupby("status")
        .agg(
            n_orders=("order_id", "count"),
            total_abs_quantity=("quantity", lambda x: np.abs(x).sum())
        )
        .reset_index()
    )

order_status_summary

## 15. Position ledger and exposure history

The NAV log records cash, market value, gross exposure, net exposure, and stress labels at every event timestamp.

In [ ]:
exposure_history = event_nav[[
    "cash", "market_value", "nav", "gross_exposure", "net_exposure",
    "long_exposure", "short_exposure", "regime", "stress_type"
]].copy()

exposure_history["gross_exposure_pct_nav"] = exposure_history["gross_exposure"] / exposure_history["nav"]
exposure_history["net_exposure_pct_nav"] = exposure_history["net_exposure"] / exposure_history["nav"]

exposure_report = pd.DataFrame([{
    "mean_gross_exposure_pct_nav": exposure_history["gross_exposure_pct_nav"].mean(),
    "p95_gross_exposure_pct_nav": exposure_history["gross_exposure_pct_nav"].quantile(0.95),
    "max_gross_exposure_pct_nav": exposure_history["gross_exposure_pct_nav"].max(),
    "mean_net_exposure_pct_nav": exposure_history["net_exposure_pct_nav"].mean(),
    "min_cash": exposure_history["cash"].min(),
    "final_cash": exposure_history["cash"].iloc[-1],
    "final_nav": exposure_history["nav"].iloc[-1],
}])

exposure_report

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(exposure_history.index, exposure_history["gross_exposure_pct_nav"], label="gross exposure / NAV")
plt.plot(exposure_history.index, exposure_history["net_exposure_pct_nav"], label="net exposure / NAV")
plt.axhline(config.max_gross_exposure, linestyle="--", label="max gross")
plt.title("Event-Driven Exposure History")
plt.xlabel("Timestamp")
plt.ylabel("Exposure")
plt.legend()
plt.show()

## 16. Reconciliation against vectorized approximation

We build a rough vectorized approximation from the event strategy’s rebalance timestamps.

This will not match exactly because the event engine uses:

- bid/ask fills;
- latency;
- partial fills;
- limit order expiry;
- cash accounting.

But it provides a useful sanity reference.

In [ ]:
def build_vectorized_target_from_event_strategy(prices, regime_info, config):
    target = pd.DataFrame(0.0, index=prices.index, columns=prices.columns)

    for t, timestamp in enumerate(prices.index):
        if t % config.rebalance_minutes == 0:
            current_regime = int(regime_info.loc[timestamp, "regime"])
            target.loc[timestamp] = generate_signal_weights(timestamp, prices.iloc[:t+1], config, current_regime)
        else:
            target.loc[timestamp] = np.nan

    return target.ffill().fillna(0.0)

def vectorized_intraday_reference(prices, target_weights, config):
    returns_mid = prices.pct_change().fillna(0.0)
    held = target_weights.shift(config.order_latency_minutes).fillna(0.0)
    gross = (held * returns_mid).sum(axis=1)
    turnover = held.diff().abs().sum(axis=1).fillna(held.abs().sum(axis=1))
    cost = turnover * (config.base_slippage_bps + 1.0) / 10000.0
    net = gross - cost
    nav = config.initial_cash * (1 + net).cumprod()
    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": held.abs().sum(axis=1),
        "net_exposure": held.sum(axis=1),
    })

vectorized_target = build_vectorized_target_from_event_strategy(prices, regime_info, config)
vectorized_reference = vectorized_intraday_reference(prices, vectorized_target, config)

reconciliation = pd.DataFrame({
    "event_nav": event_nav["nav"],
    "vectorized_reference_nav": vectorized_reference["nav"],
})
reconciliation["nav_difference"] = reconciliation["event_nav"] - reconciliation["vectorized_reference_nav"]
reconciliation["nav_difference_pct_initial"] = reconciliation["nav_difference"] / config.initial_cash

reconciliation.tail()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(reconciliation.index, reconciliation["event_nav"], label="event-driven")
plt.plot(reconciliation.index, reconciliation["vectorized_reference_nav"], label="vectorized reference", linestyle="--")
plt.title("Event-Driven vs Vectorized Reference NAV")
plt.xlabel("Timestamp")
plt.ylabel("NAV")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(reconciliation.index, reconciliation["nav_difference_pct_initial"])
plt.title("Event-Driven Minus Vectorized Reference")
plt.xlabel("Timestamp")
plt.ylabel("Difference / initial capital")
plt.show()

## 17. Stress-period diagnostics

The event engine makes it easy to inspect stress-period behaviour and fill quality during stress.

In [ ]:
def event_stress_report(event_nav, fills, regime_info):
    nav_joined = event_nav.join(regime_info, rsuffix="_regime")

    stress_summary = (
        nav_joined
        .groupby("stress_type")
        .agg(
            n_periods=("return", "count"),
            mean_return=("return", "mean"),
            vol=("return", "std"),
            worst_period=("return", "min"),
            mean_gross_exposure=("gross_exposure", "mean"),
        )
        .reset_index()
    )

    if fills.empty:
        fill_stress = pd.DataFrame()
    else:
        fill_stress = fills.copy()
        fill_stress = fill_stress.join(regime_info, on="timestamp")
        fill_stress = (
            fill_stress
            .groupby("stress_type")
            .agg(
                n_fills=("fill_id", "count"),
                abs_notional=("abs_notional", "sum"),
                mean_effective_spread_bps=("effective_spread_bps", "mean"),
                total_commission=("commission", "sum"),
            )
            .reset_index()
        )

    return stress_summary, fill_stress

event_stress_summary, fill_stress_summary = event_stress_report(event_nav, fills, regime_info)

event_stress_summary, fill_stress_summary

## 18. Event audit trail

The engine stores:

- event log;
- order log;
- fill log;
- financing log;
- NAV log.

This is crucial for debugging and research review.

In [ ]:
event_log = event_results["events"]
order_log = event_results["orders"]
fill_log = event_results["fills"]
financing_log = event_results["financing"]

audit_counts = pd.Series({
    "n_market_events": int((event_log["event_type"] == "MARKET").sum()) if not event_log.empty else 0,
    "n_signal_events": int((event_log["event_type"] == "SIGNAL").sum()) if not event_log.empty else 0,
    "n_order_records": int(len(order_log)),
    "n_fill_records": int(len(fill_log)),
    "n_nav_records": int(len(event_nav)),
    "n_financing_records": int(len(financing_log)),
})

audit_counts

## 19. Governance flags

A strategy should be reviewed if:

1. NAV becomes non-positive;
2. gross exposure breaches limit;
3. many orders expire;
4. effective spread is too high;
5. event-driven and vectorized results diverge sharply;
6. cash becomes too negative;
7. stress-period losses are severe;
8. fill count is unexpectedly low.

In [ ]:
expired_orders = 0
if not latest_orders.empty and "status" in latest_orders.columns:
    expired_orders = int((latest_orders["status"] == "EXPIRED").sum())

n_orders = int(latest_orders["order_id"].nunique()) if not latest_orders.empty else 0
expired_rate = expired_orders / n_orders if n_orders else 0.0

mean_eff_spread = fills["effective_spread_bps"].mean() if not fills.empty else np.nan
p95_eff_spread = fills["effective_spread_bps"].quantile(0.95) if not fills.empty else np.nan
nav_divergence = abs(reconciliation["nav_difference_pct_initial"].iloc[-1])
worst_return = event_nav["return"].min()
max_gross_pct = exposure_history["gross_exposure_pct_nav"].max()
min_cash = exposure_history["cash"].min()

governance_flags = pd.DataFrame([{
    "strategy": "event_driven_strategy",
    "final_nav": event_nav["nav"].iloc[-1],
    "min_nav": event_nav["nav"].min(),
    "max_gross_exposure_pct_nav": max_gross_pct,
    "expired_order_rate": expired_rate,
    "mean_effective_spread_bps": mean_eff_spread,
    "p95_effective_spread_bps": p95_eff_spread,
    "event_vs_vectorized_final_nav_divergence_pct_initial": nav_divergence,
    "worst_period_return": worst_return,
    "min_cash": min_cash,
    "n_fills": len(fills),
    "flag_nav_nonpositive": bool(event_nav["nav"].min() <= 0),
    "flag_gross_exposure_breach": bool(max_gross_pct > config.max_gross_exposure + 0.05),
    "flag_expired_orders_above_30pct": bool(expired_rate > 0.30),
    "flag_p95_spread_above_25bps": bool(p95_eff_spread > 25) if np.isfinite(p95_eff_spread) else True,
    "flag_event_vectorized_divergence_above_5pct": bool(nav_divergence > 0.05),
    "flag_cash_below_minus_20pct_capital": bool(min_cash < -0.20 * config.initial_cash),
    "flag_worst_period_below_minus_3pct": bool(worst_return < -0.03),
    "flag_few_fills": bool(len(fills) < 10),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_nav_nonpositive",
        "flag_gross_exposure_breach",
        "flag_expired_orders_above_30pct",
        "flag_p95_spread_above_25bps",
        "flag_event_vectorized_divergence_above_5pct",
        "flag_cash_below_minus_20pct_capital",
        "flag_worst_period_below_minus_3pct",
        "flag_few_fills",
    ]
].any(axis=1)

governance_flags

## 20. Audit checks

The event-driven engine should pass basic accounting checks:

1. NAV equals cash plus marked positions;
2. fills reference valid order IDs;
3. commissions are non-negative;
4. fill prices are finite and positive;
5. positions are finite;
6. NAV is finite;
7. order timestamps precede eligible timestamps;
8. event log is chronological.

In [ ]:
audit_rows = []

nav_recalc = event_nav["cash"] + event_nav["market_value"]
nav_diff = (nav_recalc - event_nav["nav"]).abs().max()
audit_rows.append({
    "check": "nav_equals_cash_plus_market_value",
    "value": float(nav_diff),
    "passed": bool(nav_diff < 1e-8),
})

if not fills.empty and not orders.empty:
    valid_order_ids = set(orders["order_id"])
    fill_order_ids_valid = set(fills["order_id"]).issubset(valid_order_ids)
else:
    fill_order_ids_valid = True
audit_rows.append({
    "check": "fills_reference_valid_orders",
    "value": bool(fill_order_ids_valid),
    "passed": bool(fill_order_ids_valid),
})

commissions_nonnegative = bool((fills["commission"] >= 0).all()) if not fills.empty else True
audit_rows.append({
    "check": "commissions_nonnegative",
    "value": commissions_nonnegative,
    "passed": commissions_nonnegative,
})

fill_prices_positive = bool((fills["fill_price"] > 0).all()) if not fills.empty else True
audit_rows.append({
    "check": "fill_prices_positive",
    "value": fill_prices_positive,
    "passed": fill_prices_positive,
})

positions_finite = bool(np.isfinite(event_results["portfolio"].positions.to_numpy()).all())
audit_rows.append({
    "check": "final_positions_finite",
    "value": positions_finite,
    "passed": positions_finite,
})

nav_finite = bool(np.isfinite(event_nav["nav"]).all())
audit_rows.append({
    "check": "nav_finite",
    "value": nav_finite,
    "passed": nav_finite,
})

if not orders.empty and "eligible_time" in orders.columns:
    order_time_ok = bool((orders["eligible_time"] >= orders["timestamp"]).all())
else:
    order_time_ok = True
audit_rows.append({
    "check": "order_eligible_time_after_timestamp",
    "value": order_time_ok,
    "passed": order_time_ok,
})

event_chronological = bool(event_log["timestamp"].is_monotonic_increasing) if not event_log.empty else True
audit_rows.append({
    "check": "event_log_chronological",
    "value": event_chronological,
    "passed": event_chronological,
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 21. Practical checklist for event-driven backtesting

Before trusting an event-driven simulation:

1. **Market data clock**  
   Are all data events timestamped correctly?

2. **Signal timing**  
   Does the strategy use only past and current available data?

3. **Order latency**  
   Are orders delayed before they can fill?

4. **Order type rules**  
   Do market and limit orders behave differently?

5. **Liquidity limits**  
   Are fills capped by volume or displayed size?

6. **Partial fills**  
   Can orders fill gradually?

7. **Cash and positions**  
   Are fills the only thing that changes positions?

8. **Commissions and slippage**  
   Are costs charged at fill time?

9. **Open orders**  
   Are expired and partially filled orders handled?

10. **Reconciliation**  
   Can event-driven results be reconciled to a simpler vectorized approximation?

11. **Audit trail**  
   Can every fill be traced back to an order and timestamp?

12. **Stress behaviour**  
   Does the strategy behave sensibly when spreads widen and volatility rises?

## 22. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/event_driven_backtest_engine_v1")
output_dir.mkdir(parents=True, exist_ok=True)

# Flatten L1 data for saving.
l1_frames = []
for asset, df in l1.items():
    temp = df.copy()
    temp["asset"] = asset
    temp["timestamp"] = temp.index
    l1_frames.append(temp.reset_index(drop=True))
l1_panel = pd.concat(l1_frames, ignore_index=True)

paths = {
    "l1_panel": output_dir / "l1_panel.csv",
    "prices": output_dir / "prices.csv",
    "returns": output_dir / "returns.csv",
    "regime_info": output_dir / "regime_info.csv",
    "metadata": output_dir / "metadata.csv",
    "nav": output_dir / "event_nav.csv",
    "orders": output_dir / "order_log.csv",
    "fills": output_dir / "fill_log.csv",
    "events": output_dir / "event_log.csv",
    "financing": output_dir / "financing_log.csv",
    "event_performance": output_dir / "event_performance.csv",
    "fill_summary": output_dir / "fill_summary.csv",
    "fill_asset_summary": output_dir / "fill_asset_summary.csv",
    "order_status_summary": output_dir / "order_status_summary.csv",
    "exposure_history": output_dir / "exposure_history.csv",
    "exposure_report": output_dir / "exposure_report.csv",
    "vectorized_target": output_dir / "vectorized_reference_target_weights.csv",
    "vectorized_reference": output_dir / "vectorized_reference.csv",
    "reconciliation": output_dir / "event_vectorized_reconciliation.csv",
    "event_stress_summary": output_dir / "event_stress_summary.csv",
    "fill_stress_summary": output_dir / "fill_stress_summary.csv",
    "audit_counts": output_dir / "audit_counts.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

l1_panel.to_csv(paths["l1_panel"], index=False)
prices.to_csv(paths["prices"])
returns.to_csv(paths["returns"])
regime_info.to_csv(paths["regime_info"])
metadata.to_csv(paths["metadata"], index=False)
event_nav.to_csv(paths["nav"])
orders.to_csv(paths["orders"], index=False)
fills.to_csv(paths["fills"], index=False)
event_log.to_csv(paths["events"], index=False)
financing_log.to_csv(paths["financing"])
event_performance.to_csv(paths["event_performance"], index=False)
fill_summary.to_csv(paths["fill_summary"], index=False)
fill_asset_summary.to_csv(paths["fill_asset_summary"], index=False)
order_status_summary.to_csv(paths["order_status_summary"], index=False)
exposure_history.to_csv(paths["exposure_history"])
exposure_report.to_csv(paths["exposure_report"], index=False)
vectorized_target.to_csv(paths["vectorized_target"])
vectorized_reference.to_csv(paths["vectorized_reference"])
reconciliation.to_csv(paths["reconciliation"])
event_stress_summary.to_csv(paths["event_stress_summary"], index=False)
fill_stress_summary.to_csv(paths["fill_stress_summary"], index=False)
audit_counts.to_frame("value").to_csv(paths["audit_counts"])
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "event_driven_backtest_engine_outputs",
    "schema_version": "event_driven_backtest_engine_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "assets": assets,
    "n_timestamps": len(prices),
    "n_orders": int(len(orders)),
    "n_fills": int(len(fills)),
    "final_nav": float(event_nav["nav"].iloc[-1]),
    "event_performance": event_performance.to_dict(orient="records"),
    "fill_summary": fill_summary.to_dict(orient="records") if not fill_summary.empty else [],
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "audit_counts": audit_counts.to_dict(),
    "notes": [
        "The engine processes market data, signal, order, fill, financing and NAV events.",
        "Orders have latency before they are eligible for execution.",
        "Limit orders can expire unfilled.",
        "Market and limit orders use bid/ask prices and liquidity caps.",
        "Positions and cash change only through fills and financing accrual.",
        "Event-driven results are reconciled against a simpler vectorized reference.",
        "This is a compact research engine, not a production execution simulator."
    ]
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["nav"], paths["fills"], paths["event_performance"], paths["governance_flags"], paths["manifest"]

## 23. Limitations

### Synthetic L1 data

The L1 data is simulated. Real L1 data has irregular timestamps, crossed markets, locked markets, bad ticks, auctions, halts, and exchange-specific rules.

### Simplified order book

Only top-of-book bid/ask and size are modelled. There is no queue priority, depth, cancellations, hidden liquidity, or market impact.

### Simplified latency

Latency is deterministic. Real latency is random and venue/order-type dependent.

### Simplified limit orders

Limit order fills are rule-based. Real limit orders depend on queue position and trade-through behaviour.

### No corporate actions or futures rolls

This engine is intraday and simplified. Production systems need contract multipliers, margin, rolls, splits, dividends, and cash events.

### No portfolio optimiser

The strategy is a test harness, not a sophisticated alpha model.

### No live broker integration

This is a research simulator, not an execution system.

## 24. Research frontier and extensions

Important extensions include:

1. queue-position modelling;
2. full limit-order-book depth;
3. stochastic latency;
4. order cancellation and replacement logic;
5. market impact;
6. participation algorithms;
7. TWAP/VWAP execution;
8. futures margin and contract multipliers;
9. auction and halt handling;
10. exchange calendar support;
11. tick-size constraints;
12. live shadow-mode reconciliation;
13. Chinese futures night-session support with L1 bid/ask data and price-limit handling.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_03_transaction_costs_slippage_latency**  
   Build a deeper execution-cost model.

2. **05_04_walk_forward_hedge_validation**  
   Validate hedge ratios with event-aware fills.

3. **05_08_strategy_capacity_and_market_impact**  
   Turn the capacity proxy into a real impact model.

4. **05_09_event_driven_backtest_framework**  
   Extend this compact engine into a more modular framework.

5. **05_11_live_shadow_mode_monitoring**  
   Compare research orders with paper/live execution.

6. **06_04_order_management_system_simulator**  
   Move into market microstructure systems.

## 26. Summary

This notebook implemented an event-driven backtest engine.

It showed how to:

1. simulate intraday L1 bid/ask data;
2. define market, signal, order, and fill events;
3. build a portfolio state object;
4. generate signal-driven target weights;
5. convert targets into orders;
6. simulate latency;
7. simulate market and limit order fills;
8. handle partial fills and order expiry;
9. update cash and positions from fills;
10. accrue borrow and financing costs;
11. record NAV and exposure history;
12. analyse fill quality;
13. analyse order lifecycle;
14. reconcile event-driven results with a vectorized reference;
15. inspect stress-period behaviour;
16. create governance flags and audit checks;
17. save event logs, order logs, fill logs, and manifests.

The key computational takeaway:

> Event-driven backtesting is stateful accounting over an ordered stream of events.

The key financial takeaway:

> A strategy is not just a desired weight vector; it is a sequence of orders that may arrive late, fill partially, pay spread, consume liquidity, and change cash and risk one fill at a time.

## 27. Further reading

- Chan, *Algorithmic Trading.*
- Cartea, Jaimungal, and Penalva, *Algorithmic and High-Frequency Trading.*
- López de Prado, *Advances in Financial Machine Learning.*
- Harris, *Trading and Exchanges.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Institutional literature on transaction-cost analysis, order simulation, market impact, and live shadow trading.